# Velocity vs Spacings

X Axis: Machine Velocity 

Y Axis: Horizontal line spacing

Purpose: Inspect half toning / darkness of each square.

In [2]:
import gcode
import numpy as np
from typing import List

In [3]:
def init(**kwargs) -> gcode.GCode:
    prog = gcode.GCode(**kwargs)
    prog.G21()
    prog.G90()
    # Zero on the K30 cross hairs.
    prog.G92(X=0, Y=0)
    # Set motion velocity.
    prog.G1(F=1000)
    prog.G0(F=1000)

    prog.M3(S=0)
    # Cross hairs offset.
    prog.G1(X=16.5, Y=1)
    prog.G92(X=0, Y=0)

    # Offset for axes labels
    prog.G1(X=10, Y=10)
    prog.G92(X=0, Y=0)

    prog.M5()
    return prog

Draw the Axis

In [4]:
def add_axes():
    # X Axis
    prog = gcode.GCode()
    prog += gcode.Line(
        points=gcode.hline(X0=0, Xf=150, Y=0), power=100, feed=1000, dynamic_power=False
    )
    prog += gcode.Line(
        points=gcode.vline(Y0=0, Yf=75, X=0), power=100, feed=1000, dynamic_power=False
    )
    # Offset to start drawing rectangles.
    prog.G1(X=5, Y=5)
    # Re-zero for the experiment
    prog.G92(X=0, Y=0)

    return prog

In [5]:
square_size = 10
speeds = np.round(np.linspace(500, 2000, 10), 0)
spacings = [0.1, 0.2, 0.3, 0.4, 0.5]
speeds

array([ 500.,  667.,  833., 1000., 1167., 1333., 1500., 1667., 1833.,
       2000.])

In [6]:
prog = init()
prog += add_axes()

Y = 0
for spacing_idx, spacing in enumerate(spacings):
    X = 0
    for speed_idx, speed in enumerate(speeds):
        for y_idx, y in enumerate(np.arange(0, 10 + spacing, spacing)):
            if np.mod(y_idx, 2) == 0:
                prog += gcode.Line(
                    points=gcode.hline(X0=X + square_size, Xf=X, Y=Y + y),
                    power=100,
                    feed=speed,
                    dynamic_power=False,
                )
            else:
                prog += gcode.Line(
                    points=gcode.hline(X0=X, Xf=X + square_size, Y=Y + y),
                    power=100,
                    feed=speed,
                    dynamic_power=False,
                )
        X += square_size
        X += 5
    Y += square_size
    Y += 5

prog.M5()
prog.G1(X=-10, Y=-10, F=2000)
prog.G92(X=0, Y=0)

prog.save("/tmp/Velocity_vs_Spacing.ngc")

In [7]:
# Increase velocity to decrease burning.

In [8]:
speeds = np.linspace(1000, 3000, 9)
spacings = [0.5, 0.75, 1, 1.25, 1.5, 2]
speeds

array([1000., 1250., 1500., 1750., 2000., 2250., 2500., 2750., 3000.])

In [9]:
prog = init()
prog += add_axes()

Y = 0
for spacing_idx, spacing in enumerate(spacings):
    X = 0
    for speed_idx, speed in enumerate(speeds):
        for y_idx, y in enumerate(np.arange(0, 10 + spacing, spacing)):
            if np.mod(y_idx, 2) == 0:
                prog += gcode.Line(
                    points=gcode.hline(X0=X + square_size, Xf=X, Y=Y + y),
                    power=100,
                    feed=speed,
                    dynamic_power=False,
                )
            else:
                prog += gcode.Line(
                    points=gcode.hline(X0=X, Xf=X + square_size, Y=Y + y),
                    power=100,
                    feed=speed,
                    dynamic_power=False,
                )
        X += square_size
        X += 5
    Y += square_size
    Y += 5

prog.M5()
prog.G1(X=-10, Y=-10, F=2000)
prog.G92(X=0, Y=0)

prog.save("/tmp/Velocity_vs_Spacing2.ngc")

# Spacing Tests 

Test various spacings of the laser bottom to the work surface. Should be set to 8mm to mark on the surface. Increasing the height widens the beam.

In [22]:
import numpy as np
from typing import List, Union
import gcode
import numpy as np
from typing import List, Union
import gcode


def velocity_vs_spacing_squares(
    speeds: List[float],
    spacings: List[float],
    square_size: Union[int, float] = 10,
    spare_spacing: Union[int, float] = 2.5,
) -> gcode.GCode:
    """
    Generates G-code for engraving squares with varying speeds and spacings.

    Parameters
    ----------
    speeds : List[float]
        List of feed rates for the engraving in units per minute.
    spacings : List[float]
        List of spacings between the lines in units.
    square_size : Union[int, float], optional
        The size of each square in units, by default 10.

    Returns
    -------
    None
        This function does not return any values but generates G-code instructions.

    Notes
    -----
    This function engraves squares by varying the speed and spacing, and outputs the G-code to the machine.
    The engravings are performed line-by-line using horizontal movements. The function also positions the machine
    head at the origin upon completion.

    Examples
    --------
    >>> speeds = [100, 200, 300]
    >>> spacings = [1, 2, 3]
    >>> velocity_vs_spacing_squares(speeds, spacings, square_size=15.5)
    """

    Y = 0
    prog = gcode.GCode()

    for spacing_idx, spacing in enumerate(spacings):
        X = 0
        for speed_idx, speed in enumerate(speeds):
            for y_idx, y in enumerate(np.arange(0, 10 + spacing, spacing)):
                if np.mod(y_idx, 2) == 0:
                    prog += gcode.Line(
                        points=gcode.hline(X0=X + square_size, Xf=X, Y=Y + y),
                        power=100,
                        feed=speed,
                        dynamic_power=False,
                    )
                else:
                    prog += gcode.Line(
                        points=gcode.hline(X0=X, Xf=X + square_size, Y=Y + y),
                        power=100,
                        feed=speed,
                        dynamic_power=False,
                    )
            X += square_size + spare_spacing
        Y += square_size + spare_spacing

    prog.M5()
    prog.G1(X=-10, Y=-10, F=2000)
    prog.G92(X=0, Y=0)
    return prog

In [24]:
speeds = np.linspace(750, 3000, 10)
print(f"Speeds: {speeds}")
spacings = [0.5, 1, 1.5, 2]
print(f"Spacings: {spacings}")

Speeds: [ 750. 1000. 1250. 1500. 1750. 2000. 2250. 2500. 2750. 3000.]
Spacings: [0.5, 1, 1.5, 2]


In [50]:
prog = init()
prog += add_axes()
prog += velocity_vs_spacing_squares(speeds, spacings)

prog.save("/tmp/velocity_vs_spacing_vs_offset.gcode")

In [49]:
speeds = np.linspace(250, 1000, 11)
print(f"Speeds: {speeds}")
spacings = [0.25, 0.5, 0.75, 1, 1.5]
print(f"Spacings: {spacings}")

prog = init()
prog += add_axes()
prog += velocity_vs_spacing_squares(speeds, spacings)

prog.save("/tmp/velocity_vs_spacing_vs_offset.gcode")

Speeds: [ 250.  325.  400.  475.  550.  625.  700.  775.  850.  925. 1000.]
Spacings: [0.25, 0.5, 0.75, 1, 1.5]


In [53]:
speeds = list(np.linspace(250, 1000, 6))
speeds.append(1500)
speeds.append(2000)
print(f"Speeds: {speeds}")
spacings = [0.5, 0.75, 1, 1.5]
print(f"Spacings: {spacings}")

prog = init()
prog += add_axes()
prog += velocity_vs_spacing_squares(speeds, spacings)

prog.save("/tmp/velocity_vs_spacing_vs_offset.gcode")

Speeds: [np.float64(250.0), np.float64(400.0), np.float64(550.0), np.float64(700.0), np.float64(850.0), np.float64(1000.0), 1500, 2000]
Spacings: [0.5, 0.75, 1, 1.5]


# Changing Jog Speed, high speeds in the Y axis 

In [60]:
import numpy as np
from typing import List, Union
import gcode
import numpy as np
from typing import List, Union
import gcode


def velocity_vs_spacing_squares(
    speeds: List[float],
    spacings: List[float],
    square_size: Union[int, float] = 10,
    square_spacing: Union[int, float] = 2.5,
) -> gcode.GCode:
    """
    Generates G-code for engraving squares with varying speeds and spacings.

    Parameters
    ----------
    speeds : List[float]
        List of feed rates for the engraving in units per minute.
    spacings : List[float]
        List of spacings between the lines in units.
    square_size : Union[int, float], optional
        The size of each square in units, by default 10.

    Returns
    -------
    None
        This function does not return any values but generates G-code instructions.

    Notes
    -----
    This function engraves squares by varying the speed and spacing, and outputs the G-code to the machine.
    The engravings are performed line-by-line using horizontal movements. The function also positions the machine
    head at the origin upon completion.

    Examples
    --------
    >>> speeds = [100, 200, 300]
    >>> spacings = [1, 2, 3]
    >>> velocity_vs_spacing_squares(speeds, spacings, square_size=15.5)
    """

    Y = 0
    prog = gcode.GCode()

    for spacing_idx, spacing in enumerate(spacings):
        X = 0
        for speed_idx, speed in enumerate(speeds):
            for y_idx, y in enumerate(np.arange(0, square_size + spacing, spacing)):
                prog.G0(F=500)
                prog.G1(F=500)
                if np.mod(y_idx, 2) == 0:
                    prog += gcode.Line(
                        points=gcode.hline(X0=X + square_size, Xf=X, Y=Y + y),
                        power=100,
                        feed=speed,
                        dynamic_power=False,
                    )
                else:
                    prog += gcode.Line(
                        points=gcode.hline(X0=X, Xf=X + square_size, Y=Y + y),
                        power=100,
                        feed=speed,
                        dynamic_power=False,
                    )
            X += square_size + square_spacing
        Y += square_size + square_spacing

    prog.M5()
    prog.G1(X=-10, Y=-10, F=2000)
    prog.G92(X=0, Y=0)
    return prog

In [61]:
speeds = list(np.linspace(250, 1000, 6))
speeds.append(1500)
speeds.append(2000)
print(f"Speeds: {speeds}")
spacings = [0.5, 0.75, 1, 1.5]
print(f"Spacings: {spacings}")

prog = init()
prog += add_axes()
prog += velocity_vs_spacing_squares(speeds, spacings, square_size=5, square_spacing=0)

prog.save("/tmp/velocity_vs_spacing_vs_offset2.gcode")

Speeds: [np.float64(250.0), np.float64(400.0), np.float64(550.0), np.float64(700.0), np.float64(850.0), np.float64(1000.0), 1500, 2000]
Spacings: [0.5, 0.75, 1, 1.5]
